# DeepGPR 离散梯度验证

这个最小例子分别验证相对介电常数和电导率梯度。验证时必须使用 `model_gradient_sampling_interval=1`；中心有限差分使用多个步长，以避开 float32 的舍入误差区间。

In [1]:
import torch
import sys 
from pathlib import Path
src_path=Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
import DeepGPR

torch.manual_seed(2026)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
FDTD_ORDER = 2
print("device:", DEVICE, "fdtd_order:", FDTD_ORDER)

device: cpu fdtd_order: 2


In [2]:
# 小型二维 TM 模型；源与接收器放在 CPML 内侧。
nx, ny, nt = 24, 30, 180
dx, dt = 0.02, 3.0e-11
frequency = 250.0e6
pml_width = 4

x = torch.arange(nx, device=DEVICE, dtype=torch.float32)[:, None]
y = torch.arange(ny, device=DEVICE, dtype=torch.float32)[None, :]
blob = torch.exp(-0.5 * (((x - 15.0) / 3.5) ** 2 + ((y - 17.0) / 4.5) ** 2))

er0 = torch.full((nx, ny), 4.0, device=DEVICE)
se0 = torch.full((nx, ny), 3.0e-4, device=DEVICE)
er_true = er0 + 0.35 * blob
se_true = se0 + 1.5e-4 * blob
mr = torch.ones_like(er0)

source_location = torch.tensor([[[6, 10, 0]]], dtype=torch.int32, device=DEVICE)
receiver_location = torch.tensor(
    [[[6, 14, 0], [6, 18, 0], [6, 22, 0]]], dtype=torch.int32, device=DEVICE
)
wavelet = DeepGPR.wavelet.ricker(frequency, nt, dt, 1.0 / frequency, dtype=torch.float32)
source_amplitudes = wavelet.reshape(1, nt, 1).to(DEVICE)

In [3]:
def simulate(er, se):
    # 严格梯度检查不能跳采样；异步卸载关闭后更容易定位数值问题。
    return DeepGPR.compute(
        device=DEVICE,
        dx=dx,
        dt=dt,
        source_amplitudes=source_amplitudes,
        source_location=source_location,
        receiver_location=receiver_location,
        er=er,
        se=se,
        mr=mr,
        pmlthick=pml_width,
        source_direction=2,
        reciever_direction=2,
        model_gradient_sampling_interval=1,
        use_async_offload=False,
        fdtd_order=FDTD_ORDER,
        mode=2,
    )[-1]

with torch.no_grad():
    observed = simulate(er_true, se_true)
data_scale = observed.detach().abs().max().clamp_min(1.0e-12)

def objective(er, se):
    residual = (simulate(er, se) - observed) / data_scale
    return 0.5 * residual.square().sum()

In [4]:
# 一次伴随传播同时得到 epsilon_r 与 sigma 梯度。
er = er0.detach().clone().requires_grad_(True)
se = se0.detach().clone().requires_grad_(True)
loss = objective(er, se)
loss.backward()

assert er.grad is not None and se.grad is not None
assert torch.isfinite(er.grad).all() and torch.isfinite(se.grad).all()
# CPML 系数在伴随中视为固定量，因此方向扰动只放在物理内部，不扰动 CPML 参考介质。
interior = torch.zeros_like(er, dtype=torch.bool)
interior[pml_width:-pml_width, pml_width:-pml_width] = True
masked_grad_er = torch.where(interior, er.grad.detach(), 0.0)
masked_grad_se = torch.where(interior, se.grad.detach(), 0.0)
direction_er = masked_grad_er / masked_grad_er.norm().clamp_min(1.0e-30)
direction_se = masked_grad_se / masked_grad_se.norm().clamp_min(1.0e-30)
print("loss:", float(loss), "|grad_er|:", float(er.grad.norm()), "|grad_se|:", float(se.grad.norm()))

loss: 0.001679073553532362 |grad_er|: 0.008090969175100327 |grad_se|: 0.575254499912262


/var/folders/nm/2n89zz0x53gbk9546w6x328m0000gn/T/ipykernel_71858/110873793.py:16: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:823.)
  print("loss:", float(loss), "|grad_er|:", float(er.grad.norm()), "|grad_se|:", float(se.grad.norm()))


In [5]:
def directional_check(name, base, direction, gradient, steps):
    adjoint_value = float((gradient * direction).sum())
    rows = []
    for h in steps:
        with torch.no_grad():
            if name == "epsilon_r":
                plus = objective(base + h * direction, se.detach())
                minus = objective(base - h * direction, se.detach())
            else:
                plus = objective(er.detach(), base + h * direction)
                minus = objective(er.detach(), base - h * direction)
        finite_difference = float((plus - minus) / (2.0 * h))
        relative_error = abs(adjoint_value - finite_difference) / max(
            abs(adjoint_value), abs(finite_difference), 1.0e-30
        )
        rows.append((h, adjoint_value, finite_difference, relative_error))
        print(
            f"{name:9s} h={h:.2e} adjoint={adjoint_value:.6e} "
            f"finite_difference={finite_difference:.6e} relative_error={relative_error:.3e}"
        )
    return rows

rows_er = directional_check(
    "epsilon_r", er.detach(), direction_er, er.grad.detach(), [8.0e-2, 4.0e-2, 2.0e-2, 1.0e-2]
)
rows_se = directional_check(
    "sigma", se.detach(), direction_se, se.grad.detach(), [5.0e-4, 2.0e-4, 1.0e-4, 5.0e-5]
)

epsilon_r h=8.00e-02 adjoint=6.853325e-03 finite_difference=6.855454e-03 relative_error=3.105e-04
epsilon_r h=4.00e-02 adjoint=6.853325e-03 finite_difference=6.853910e-03 relative_error=8.533e-05
epsilon_r h=2.00e-02 adjoint=6.853325e-03 finite_difference=6.854214e-03 relative_error=1.297e-04
epsilon_r h=1.00e-02 adjoint=6.853325e-03 finite_difference=6.852229e-03 relative_error=1.599e-04
sigma     h=5.00e-04 adjoint=5.119314e-01 finite_difference=5.118852e-01 relative_error=9.023e-05
sigma     h=2.00e-04 adjoint=5.119314e-01 finite_difference=5.118595e-01 relative_error=1.404e-04
sigma     h=1.00e-04 adjoint=5.119314e-01 finite_difference=5.123013e-01 relative_error=7.220e-04
sigma     h=5.00e-05 adjoint=5.119314e-01 finite_difference=5.125744e-01 relative_error=1.254e-03


In [6]:
best_er = min(row[3] for row in rows_er)
best_se = min(row[3] for row in rows_se)
assert best_er < 0.15, f"epsilon_r gradient check failed: best relative error={best_er:.3e}"
assert best_se < 0.15, f"sigma gradient check failed: best relative error={best_se:.3e}"
print(f"PASS: epsilon_r best error={best_er:.3e}, sigma best error={best_se:.3e}")

PASS: epsilon_r best error=8.533e-05, sigma best error=9.023e-05
